*Tarea - Práctica 1.*

Integrantes:
+ Galan Resendiz Andres Roberto
+ Olivares Hernandez Argenis Daniel
+ Ramos Reyes Paulina

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
from pandas.plotting import parallel_coordinates
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE


age - edad


wtkg - peso


hemo - hemofilia


drugs - uso de drogas


karnof -  estado funcional


preanti - tratamiento previo


cd40 - CD4 inicial


cd420 - CD4 posterior


cd80 - CD8 inicial


cd820 - CD8 posterior


infected - resultado que queremos predecir

**A responder**


* ¿Qué características de los pacientes se relacionan con la infección?
* ¿Cómo se comportan conjuntamente las variables clínicas, demográficas y de tratamiento?
*  ¿Los pacientes que comienzan con valores altos de CD4 los conservan después de tiempo?
* ¿Hay alguna relación entre la edad y los niveles de CD4?
* ¿El peso influye en los patrones de las células de CD4?





# ***Paso 0. Mantenimiento de datos***

In [ ]:
df = pd.read_csv("/AIDS_Classification.csv")

df.info()

print("VALORES FALTANTES POR COLUMNA")
faltantes = df.isnull().sum()
print(faltantes)
print(f"\nValores faltantes en la base: {faltantes.sum()}")

print("DUPLICADOS")
duplicados = df.duplicated().sum()
print(f"Número de filas duplicadas: {duplicados}")

FileNotFoundError: [Errno 2] No such file or directory: '/AIDS_Classification.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ***Paso 1. Estadísticas***



In [ ]:
# ESTADISTICAS DE LAS VARIBALES QUE SON CONTINUAS
vars_num = ['age', 'wtkg', 'cd40', 'cd420', 'cd80', 'cd820']
print(df[vars_num].describe().round(2))


In [ ]:
df[vars_num].hist(figsize=(14, 12),bins=30)
plt.tight_layout()
plt.show()

In [ ]:
#NO PONER EN REPORTE, SOLO ANALIZAR
sns.countplot(data=df, x="infected")
plt.title("Distribución de infectados")
plt.xlabel("Infected")
plt.ylabel("Observaciones totales")
plt.show()

In [ ]:
# CD4 INICIAL vs A LAS 20 SEMANAS

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CD4 INICIAL
sns.histplot(data=df, x='cd40', hue='infected', kde=True, ax=axes[0], palette='viridis')
axes[0].set_title('CD4 Inicial')
axes[0].set_xlabel('Células CD4 iniciales')

# CD4 A LAS 20 SEMANAS
sns.histplot(data=df, x='cd420', hue='infected', kde=True, ax=axes[1], palette='viridis')
axes[1].set_title('CD4 a las 20 semanas')
axes[1].set_xlabel('Células CD4 a las 20 semanas')

plt.tight_layout()
plt.show()

In [ ]:
# PARA OUTLIERS

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='infected', y='cd80', palette='viridis')
plt.title('CD8 iniciales según tiempo transcurrido')
plt.xlabel('Progresión a SIDA (0 = No, 1 = Sí)')
plt.ylabel('Conteo Celular')
plt.show()

# ***Paso 2. Análisis Multivariado***

In [ ]:
# HEATMAP
correl = df[vars_num].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correl, annot=True, fmt=".2f", center=0, cmap = 'viridis')
plt.title("Matriz de correlación")
plt.show()

In [ ]:
# SCATTER

# RELACIONES ENTRE VARIABLES 0, 1
df_sample = df.sample(n=5, random_state=42) #JUGAR CON n
sns.pairplot(df_sample[vars_num + ['infected']], hue='infected', palette={0: 'blue', 1: 'green'}, corner=True)
plt.suptitle('Matriz de Dispersión por Estado de Infección', y=1.02)
plt.show()

In [ ]:
#ANDREWS
from pandas.plotting import andrews_curves
df_sample = df.sample(n=200, random_state=42)
plt.figure(figsize=(10, 6))
andrews_curves(df_sample, class_column="infected", colormap="viridis")
plt.title("Curvas de Andrews")
plt.xlabel("t")
plt.ylabel("f(t)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# COORDENADAS PARALELAS
from sklearn.preprocessing import MinMaxScaler
cols_interes = ["age", "wtkg", "cd40", "cd420", "cd80", "cd820", "infected"]
df_par = df[cols_interes].copy().sample(n=200, random_state=42)
features = [c for c in cols_interes if c != "infected"]
scaler = MinMaxScaler()
df_par[features] = scaler.fit_transform(df_par[features])

plt.figure(figsize=(12, 6))
parallel_coordinates(
    df_par, class_column="infected", color=["#1f77b4", "#ff7f0e"], alpha=0.5
)
plt.title("Gráfico de Coordenadas Paralelas (Normalizado)")
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# t-SNE
from sklearn.preprocessing import StandardScaler
X = df.drop(columns=["infected"])
y = df["infected"]
X_scaled = StandardScaler().fit_transform(X)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)
plt.figure(figsize=(9, 6))
sns.scatterplot(
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    hue=y,
    palette="coolwarm",
    alpha=0.7,
    edgecolor="none",
)
plt.title("Visualización t-SNE (2D)")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.legend(title="Infected")
plt.grid(True, alpha=0.2)
plt.show()